In [1]:
# ============================================================
# SetStop-KGQA Notebook 04
# Untouched Test and Online Search Evaluation
# Oracle Answer-Recognition Setting
# Complete single-cell Colab code
# ============================================================

!pip -q install -U \
    transformers \
    peft \
    accelerate \
    bitsandbytes \
    scikit-learn \
    safetensors

# ============================================================
# 1. IMPORTS
# ============================================================

import collections
import gc
import hashlib
import json
import math
import random
import re
import time

from pathlib import Path

import numpy as np
import torch
import transformers

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from peft import PeftModel
from google.colab import drive


# ============================================================
# 2. CLEAN RUNTIME AND MOUNT DRIVE
# ============================================================

for variable_name in [
    "model",
    "base_model",
    "tokenizer",
]:
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

drive.mount("/content/drive")


# ============================================================
# 3. CONFIGURATION
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path(
    "/content/drive/MyDrive/SetStop_KGQA"
)

DATA_DIR = (
    ROOT
    / "data/extracted/webqsp"
)

TEST_STATE_PATH = (
    ROOT
    / "data/processed/"
      "webqsp_test_states_v2_full.jsonl"
)

MODEL_PATH = (
    ROOT
    / "models/"
      "setstop_qwen_gain_controller_v1"
)

THRESHOLD_PATH = (
    ROOT
    / "outputs/controller_training_v1/"
      "dev_stop_threshold_v1.json"
)

OUTPUT_ROOT = (
    ROOT
    / "outputs/test_evaluation_v1"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

MAX_ANSWERS = 50
MAX_HOPS = 3
MAX_ACTIONS = 12

# Match the candidate-set size used during training.
MAX_FRONTIER_CANDIDATES = 16

INFERENCE_BATCH_SIZE = 64

GAIN_VALUES = np.asarray(
    [0.0, 1.0, 2.0, 3.5],
    dtype=np.float64
)

LABEL_NAMES = [
    "gain_0",
    "gain_1",
    "gain_2",
    "gain_3_plus",
]

ID_TO_LABEL = {
    0: "0",
    1: "1",
    2: "2",
    3: "3_plus",
}

LABEL_TO_ID = {
    value: key
    for key, value in ID_TO_LABEL.items()
}

for required_path in [
    DATA_DIR / "test_simple.json",
    DATA_DIR / "entities.txt",
    DATA_DIR / "relations.txt",
    TEST_STATE_PATH,
    MODEL_PATH,
    THRESHOLD_PATH,
]:
    assert required_path.exists(), (
        f"Missing required input:\n"
        f"{required_path}"
    )

threshold_payload = json.loads(
    THRESHOLD_PATH.read_text(
        encoding="utf-8"
    )
)

STOP_THRESHOLD = float(
    threshold_payload["selected_threshold"]
)

assert abs(STOP_THRESHOLD - 0.71) < 1e-6, (
    f"Unexpected threshold: {STOP_THRESHOLD}"
)

print("=" * 90)
print("SETSTOP-KGQA UNTOUCHED TEST EVALUATION")
print("=" * 90)
print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Frozen threshold:", STOP_THRESHOLD)
print("Maximum actions:", MAX_ACTIONS)
print(
    "Maximum frontier candidates:",
    MAX_FRONTIER_CANDIDATES
)

assert torch.cuda.is_available(), (
    "Select a T4 GPU runtime."
)

GPU_NAME = torch.cuda.get_device_name(0)

print("GPU:", GPU_NAME)
print("=" * 90)


# ============================================================
# 4. DATA HELPERS
# ============================================================

def read_vocabulary(path):
    with open(
        path,
        encoding="utf-8"
    ) as file:
        return [
            line.rstrip("\n")
            for line in file
        ]


ENTITY_NAMES = read_vocabulary(
    DATA_DIR / "entities.txt"
)

RELATION_NAMES = read_vocabulary(
    DATA_DIR / "relations.txt"
)

ENTITY_TO_ID = {
    entity_name: index
    for index, entity_name
    in enumerate(ENTITY_NAMES)
}

RELATION_TO_ID = {
    relation_name: index
    for index, relation_name
    in enumerate(RELATION_NAMES)
}


def iter_jsonl(path):
    with open(
        path,
        encoding="utf-8"
    ) as file:

        for line_number, line in enumerate(
            file,
            start=1
        ):
            if not line.strip():
                continue

            try:
                yield json.loads(line)

            except Exception as error:
                raise ValueError(
                    f"Invalid JSON at "
                    f"{path}:{line_number}"
                ) from error


def entity_index(value):
    if isinstance(value, (int, np.integer)):
        return int(value)

    if isinstance(value, str):
        return ENTITY_TO_ID.get(value)

    if isinstance(value, dict):
        for key in [
            "kb_id",
            "id",
            "entity",
        ]:
            if key in value:
                return entity_index(
                    value[key]
                )

    return None


def relation_index(value):
    if isinstance(value, (int, np.integer)):
        return int(value)

    if isinstance(value, str):
        return RELATION_TO_ID.get(value)

    return None


def get_question(example):
    for key in [
        "question",
        "text",
        "query",
    ]:
        value = example.get(key)

        if isinstance(value, str):
            return value.strip()

    raise KeyError(
        "Question field was not found"
    )


def get_topic_ids(example):
    raw_entities = example.get(
        "entities",
        example.get("topic_entities", [])
    )

    topic_ids = {
        entity_index(value)
        for value in raw_entities
    }

    return {
        value
        for value in topic_ids
        if value is not None
    }


def get_gold_ids(example):
    gold_ids = {
        entity_index(value)
        for value in example.get(
            "answers",
            []
        )
    }

    return {
        value
        for value in gold_ids
        if value is not None
    }


def get_graph(example):
    subgraph = example.get(
        "subgraph",
        {}
    )

    graph_entities = {
        entity_index(value)
        for value in subgraph.get(
            "entities",
            []
        )
    }

    graph_entities.discard(None)

    raw_edges = subgraph.get(
        "tuples",
        subgraph.get("triples", [])
    )

    edges = []

    for row in raw_edges:
        if isinstance(row, dict):
            subject = entity_index(
                row.get(
                    "subject",
                    row.get("s")
                )
            )

            relation = relation_index(
                row.get(
                    "relation",
                    row.get("r")
                )
            )

            object_entity = entity_index(
                row.get(
                    "object",
                    row.get("o")
                )
            )

        else:
            if len(row) != 3:
                continue

            subject = entity_index(row[0])
            relation = relation_index(row[1])
            object_entity = entity_index(
                row[2]
            )

        if (
            subject is None
            or relation is None
            or object_entity is None
        ):
            continue

        subject = int(subject)
        relation = int(relation)
        object_entity = int(object_entity)

        edges.append(
            (
                subject,
                relation,
                object_entity,
            )
        )

        graph_entities.add(subject)
        graph_entities.add(
            object_entity
        )

    return graph_entities, edges


def question_id(
    split,
    example,
    line_number
):
    for key in [
        "id",
        "qid",
        "question_id",
    ]:
        if key in example:
            return str(example[key])

    digest = hashlib.sha1(
        get_question(example).encode(
            "utf-8"
        )
    ).hexdigest()[:12]

    return (
        f"{split}_{line_number}_{digest}"
    )


def relation_name(relation_id):
    if 0 <= relation_id < len(
        RELATION_NAMES
    ):
        return RELATION_NAMES[
            relation_id
        ]

    return f"relation_{relation_id}"


def stable_seed(text):
    digest = hashlib.sha1(
        text.encode("utf-8")
    ).hexdigest()[:8]

    return SEED + int(digest, 16)


# ============================================================
# 5. GRAPH-SEARCH HELPERS
# ============================================================

def build_grouped_adjacency(edges):
    temporary = collections.defaultdict(
        lambda: collections.defaultdict(set)
    )

    for subject, relation, object_entity in edges:
        temporary[subject][relation].add(
            object_entity
        )

    adjacency = {}

    for source, relation_map in temporary.items():
        adjacency[source] = {
            relation: sorted(targets)
            for relation, targets
            in relation_map.items()
        }

    return adjacency


def get_frontier_branches(
    visited,
    node_depth,
    expanded_branches,
    adjacency
):
    branches = []

    for source in sorted(visited):
        source_depth = node_depth.get(
            source,
            MAX_HOPS + 1
        )

        if source_depth >= MAX_HOPS:
            continue

        for relation, targets in adjacency.get(
            source,
            {}
        ).items():
            branch_key = (
                source,
                relation
            )

            if branch_key in expanded_branches:
                continue

            branches.append({
                "source": source,
                "relation": relation,
                "targets": targets,
                "source_depth":
                    source_depth,
                "target_depth":
                    source_depth + 1,
                "target_count":
                    len(targets),
            })

    return branches


def tokenize_relation(text):
    return set(
        re.findall(
            r"[a-z0-9]+",
            text.lower().replace("_", " ")
        )
    )


def relation_overlap(
    question,
    relation_id
):
    question_tokens = tokenize_relation(
        question
    )

    relation_tokens = tokenize_relation(
        relation_name(relation_id)
    )

    return len(
        question_tokens.intersection(
            relation_tokens
        )
    )


def practical_frontier_selection(
    branches,
    question,
    maximum_candidates
):
    """
    Deterministic first-stage retrieval.

    This uses no gold answers. It prioritizes:
    1. question-relation overlap;
    2. shallow graph depth;
    3. branches returning more targets.
    """

    ranked = sorted(
        branches,
        key=lambda branch: (
            -relation_overlap(
                question,
                branch["relation"]
            ),
            branch["source_depth"],
            -branch["target_count"],
            branch["relation"],
            branch["source"],
        )
    )

    return ranked[:maximum_candidates]


def make_model_input(
    question,
    found_count,
    visited_count,
    step,
    branch,
    history_relations
):
    recent_history = (
        " -> ".join(
            history_relations[-6:]
        )
        if history_relations
        else "none"
    )

    return (
        f"Question: {question}\n"
        f"Answers found so far: {found_count}\n"
        f"Visited entities: {visited_count}\n"
        f"Search step: {step}\n"
        f"Source depth: "
        f"{branch['source_depth']}\n"
        f"Candidate relation: "
        f"{relation_name(branch['relation'])}\n"
        f"Candidate target count: "
        f"{branch['target_count']}\n"
        f"Recent relations: {recent_history}"
    )


def expand_branch(
    branch,
    visited,
    node_depth,
    expanded_branches
):
    branch_key = (
        branch["source"],
        branch["relation"],
    )

    expanded_branches.add(
        branch_key
    )

    newly_visited = set()

    for target in branch["targets"]:
        if target not in visited:
            visited.add(target)
            newly_visited.add(target)

        previous_depth = node_depth.get(
            target
        )

        if (
            previous_depth is None
            or branch["target_depth"]
            < previous_depth
        ):
            node_depth[target] = (
                branch["target_depth"]
            )

    return newly_visited


def reachable_unseen_gold(
    branch,
    adjacency,
    expanded_branches,
    unseen_gold
):
    queue = collections.deque()
    best_depth = {}

    for target in branch["targets"]:
        best_depth[target] = (
            branch["target_depth"]
        )

        queue.append(
            (
                target,
                branch["target_depth"],
            )
        )

    reachable = set()

    while queue:
        node, depth = queue.popleft()

        if node in unseen_gold:
            reachable.add(node)

        if depth >= MAX_HOPS:
            continue

        for relation, targets in adjacency.get(
            node,
            {}
        ).items():
            if (
                node,
                relation
            ) in expanded_branches:
                continue

            next_depth = depth + 1

            for target in targets:
                previous_depth = (
                    best_depth.get(target)
                )

                if (
                    previous_depth is None
                    or next_depth
                    < previous_depth
                ):
                    best_depth[target] = (
                        next_depth
                    )

                    queue.append(
                        (
                            target,
                            next_depth,
                        )
                    )

    return reachable


# ============================================================
# 6. LOAD TRAINED MODEL
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = (
        tokenizer.eos_token
    )

tokenizer.padding_side = "right"

quantization_configuration = (
    BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=
            torch.float16,
    )
)

base_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        MODEL_NAME,
        num_labels=4,
        id2label=ID_TO_LABEL,
        label2id=LABEL_TO_ID,
        quantization_config=
            quantization_configuration,
        device_map={"": 0},
        dtype=torch.float16,
    )
)

base_model.config.pad_token_id = (
    tokenizer.pad_token_id
)

model = PeftModel.from_pretrained(
    base_model,
    MODEL_PATH,
    is_trainable=False,
)

model.eval()

print("\nLoaded trained controller:")
print(MODEL_PATH)


# ============================================================
# 7. MODEL INFERENCE
# ============================================================

@torch.inference_mode()
def predict_probabilities(
    texts,
    batch_size=INFERENCE_BATCH_SIZE
):
    all_probabilities = []

    for start_index in range(
        0,
        len(texts),
        batch_size
    ):
        batch_texts = texts[
            start_index:
            start_index + batch_size
        ]

        encoded = tokenizer(
            batch_texts,
            truncation=True,
            max_length=256,
            padding=True,
            return_tensors="pt",
        )

        encoded = {
            key: value.to(model.device)
            for key, value
            in encoded.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = model(
                **encoded
            ).logits

        probabilities = torch.softmax(
            logits.float(),
            dim=-1
        )

        all_probabilities.append(
            probabilities.cpu().numpy()
        )

    return np.concatenate(
        all_probabilities,
        axis=0
    )


# ============================================================
# 8. OFFLINE UNTOUCHED TEST EVALUATION
# ============================================================

print(
    "\nRunning offline untouched-test "
    "candidate evaluation..."
)

offline_true_labels = []
offline_predictions = []

offline_state_scores = (
    collections.defaultdict(list)
)

offline_state_labels = {}

offline_candidate_count = 0
offline_state_count = 0

offline_start = time.time()

for state in iter_jsonl(
    TEST_STATE_PATH
):
    offline_state_count += 1

    candidates = state["candidates"]

    texts = [
        candidate["model_input"]
        for candidate in candidates
    ]

    probabilities = (
        predict_probabilities(texts)
    )

    true_labels = np.asarray([
        LABEL_TO_ID[
            candidate["gain_class"]
        ]
        for candidate in candidates
    ])

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    offline_true_labels.extend(
        true_labels.tolist()
    )

    offline_predictions.extend(
        predictions.tolist()
    )

    state_key = (
        state["qid"],
        int(
            state.get(
                "trajectory_id",
                0
            )
        ),
        int(state["step"]),
    )

    positive_gain_probabilities = (
        1.0 - probabilities[:, 0]
    )

    offline_state_scores[
        state_key
    ].extend(
        positive_gain_probabilities.tolist()
    )

    offline_state_labels[
        state_key
    ] = int(state["stop_label"])

    offline_candidate_count += len(
        candidates
    )

    if offline_state_count % 500 == 0:
        print(
            "Offline states:",
            offline_state_count,
            "| candidates:",
            offline_candidate_count
        )


offline_true_labels = np.asarray(
    offline_true_labels
)

offline_predictions = np.asarray(
    offline_predictions
)

test_classification_report = (
    classification_report(
        offline_true_labels,
        offline_predictions,
        labels=[0, 1, 2, 3],
        target_names=LABEL_NAMES,
        output_dict=True,
        zero_division=0,
    )
)

test_confusion_matrix = (
    confusion_matrix(
        offline_true_labels,
        offline_predictions,
        labels=[0, 1, 2, 3],
    ).tolist()
)

state_scores = []
state_true_stop = []

for state_key, scores in (
    offline_state_scores.items()
):
    state_scores.append(max(scores))
    state_true_stop.append(
        offline_state_labels[state_key]
    )

state_scores = np.asarray(
    state_scores,
    dtype=np.float64
)

state_true_stop = np.asarray(
    state_true_stop,
    dtype=np.int64
)

state_predicted_stop = (
    state_scores < STOP_THRESHOLD
).astype(np.int64)

stop_precision, stop_recall, stop_f1, _ = (
    precision_recall_fscore_support(
        state_true_stop,
        state_predicted_stop,
        average="binary",
        zero_division=0,
    )
)

offline_metrics = {
    "evaluation": (
        "untouched_test_offline_states"
    ),
    "frozen_threshold":
        STOP_THRESHOLD,
    "number_of_candidates":
        int(len(offline_true_labels)),
    "number_of_states":
        int(len(state_true_stop)),
    "candidate_accuracy": float(
        accuracy_score(
            offline_true_labels,
            offline_predictions
        )
    ),
    "candidate_balanced_accuracy": float(
        balanced_accuracy_score(
            offline_true_labels,
            offline_predictions
        )
    ),
    "candidate_macro_f1": float(
        f1_score(
            offline_true_labels,
            offline_predictions,
            average="macro",
            zero_division=0,
        )
    ),
    "candidate_weighted_f1": float(
        f1_score(
            offline_true_labels,
            offline_predictions,
            average="weighted",
            zero_division=0,
        )
    ),
    "classification_report":
        test_classification_report,
    "confusion_matrix":
        test_confusion_matrix,
    "stop_accuracy": float(
        accuracy_score(
            state_true_stop,
            state_predicted_stop
        )
    ),
    "stop_precision":
        float(stop_precision),
    "stop_recall":
        float(stop_recall),
    "stop_f1":
        float(stop_f1),
    "premature_stop_rate": float(
        np.mean(
            (state_predicted_stop == 1)
            & (state_true_stop == 0)
        )
    ),
    "over_search_rate": float(
        np.mean(
            (state_predicted_stop == 0)
            & (state_true_stop == 1)
        )
    ),
    "stop_brier_score": float(
        brier_score_loss(
            state_true_stop,
            1.0 - state_scores
        )
    ),
    "runtime_minutes": float(
        (time.time() - offline_start) / 60
    ),
}

offline_metrics_path = (
    OUTPUT_ROOT
    / "offline_test_metrics_v1.json"
)

offline_metrics_path.write_text(
    json.dumps(
        offline_metrics,
        indent=2
    ),
    encoding="utf-8",
)

print("\nOffline untouched-test results")
print(json.dumps(
    offline_metrics,
    indent=2
))


# ============================================================
# 9. LOAD TEST QUESTIONS
# ============================================================

test_examples = []

for line_number, example in enumerate(
    iter_jsonl(
        DATA_DIR / "test_simple.json"
    ),
    start=1
):
    gold_answers = get_gold_ids(
        example
    )

    topics = get_topic_ids(
        example
    )

    graph_nodes, graph_edges = (
        get_graph(example)
    )

    usable = (
        2 <= len(gold_answers)
        <= MAX_ANSWERS
        and gold_answers.issubset(
            graph_nodes
        )
        and bool(topics)
        and bool(graph_edges)
    )

    if usable:
        qid = question_id(
            "test",
            example,
            line_number
        )

        test_examples.append(
            (qid, example)
        )

print(
    "\nOnline test questions:",
    len(test_examples)
)

assert len(test_examples) == 736, (
    f"Expected 736 test questions, "
    f"found {len(test_examples)}"
)


# ============================================================
# 10. ONLINE SEARCH POLICIES
# ============================================================

def score_model_branches(
    branches,
    question,
    found_count,
    visited_count,
    step,
    history_relations
):
    model_inputs = [
        make_model_input(
            question=question,
            found_count=found_count,
            visited_count=visited_count,
            step=step,
            branch=branch,
            history_relations=
                history_relations,
        )
        for branch in branches
    ]

    probabilities = predict_probabilities(
        model_inputs
    )

    positive_probabilities = (
        1.0 - probabilities[:, 0]
    )

    expected_gains = (
        probabilities @ GAIN_VALUES
    )

    return (
        probabilities,
        positive_probabilities,
        expected_gains,
    )


def choose_oracle_branch(
    branches,
    adjacency,
    expanded_branches,
    gold_answers,
    found_answers
):
    unseen_gold = (
        gold_answers - found_answers
    )

    scored = []

    for branch in branches:
        reachable = reachable_unseen_gold(
            branch=branch,
            adjacency=adjacency,
            expanded_branches=
                expanded_branches,
            unseen_gold=unseen_gold,
        )

        scored.append(
            (
                len(reachable),
                branch
            )
        )

    scored.sort(
        key=lambda item: (
            -item[0],
            item[1]["source_depth"],
            -item[1]["target_count"],
            item[1]["relation"],
            item[1]["source"],
        )
    )

    return scored[0]


def evaluate_search_policy(
    qid,
    example,
    policy
):
    question = get_question(example)
    gold_answers = get_gold_ids(example)
    topic_entities = get_topic_ids(
        example
    )

    _, edges = get_graph(example)

    adjacency = build_grouped_adjacency(
        edges
    )

    visited = set(topic_entities)

    node_depth = {
        topic: 0
        for topic in topic_entities
    }

    expanded_branches = set()
    history_relations = []

    found_answers = (
        visited.intersection(
            gold_answers
        )
    )

    rng = random.Random(
        stable_seed(
            f"{qid}_{policy}"
        )
    )

    actions = 0
    retrieved_edges = 0
    zero_gain_streak = 0
    over_search_actions = 0

    complete_seen = (
        found_answers == gold_answers
    )

    stop_reason = None
    frontier_truncations = 0
    total_available_frontiers = 0
    maximum_stop_score = None

    while actions < MAX_ACTIONS:
        all_branches = get_frontier_branches(
            visited=visited,
            node_depth=node_depth,
            expanded_branches=
                expanded_branches,
            adjacency=adjacency,
        )

        if not all_branches:
            stop_reason = "empty_frontier"
            break

        total_available_frontiers += len(
            all_branches
        )

        if (
            len(all_branches)
            > MAX_FRONTIER_CANDIDATES
        ):
            frontier_truncations += 1

        practical_branches = (
            practical_frontier_selection(
                branches=all_branches,
                question=question,
                maximum_candidates=
                    MAX_FRONTIER_CANDIDATES,
            )
        )

        chosen_branch = None

        if policy in [
            "setstop",
            "model_fixed",
        ]:
            (
                probabilities,
                positive_probabilities,
                expected_gains,
            ) = score_model_branches(
                branches=practical_branches,
                question=question,
                found_count=
                    len(found_answers),
                visited_count=len(visited),
                step=actions,
                history_relations=
                    history_relations,
            )

            best_index = int(
                np.argmax(expected_gains)
            )

            maximum_stop_score = float(
                np.max(
                    positive_probabilities
                )
            )

            if (
                policy == "setstop"
                and maximum_stop_score
                < STOP_THRESHOLD
            ):
                stop_reason = (
                    "learned_stop"
                )
                break

            chosen_branch = (
                practical_branches[
                    best_index
                ]
            )

        elif policy == "relation_fixed":
            chosen_branch = (
                practical_branches[0]
            )

        elif policy == "fifo_fixed":
            fifo_branches = sorted(
                all_branches,
                key=lambda branch: (
                    branch["source_depth"],
                    branch["relation"],
                    branch["source"],
                )
            )

            chosen_branch = (
                fifo_branches[0]
            )

        elif policy in [
            "patience1",
            "patience2",
        ]:
            chosen_branch = (
                practical_branches[0]
            )

        elif policy == "random_fixed":
            chosen_branch = (
                all_branches[
                    rng.randrange(
                        len(all_branches)
                    )
                ]
            )

        elif policy == "oracle":
            oracle_gain, chosen_branch = (
                choose_oracle_branch(
                    branches=all_branches,
                    adjacency=adjacency,
                    expanded_branches=
                        expanded_branches,
                    gold_answers=
                        gold_answers,
                    found_answers=
                        found_answers,
                )
            )

            if oracle_gain == 0:
                stop_reason = "oracle_stop"
                break

        else:
            raise ValueError(
                f"Unknown policy: {policy}"
            )

        found_before = len(
            found_answers
        )

        was_complete_before_action = (
            found_answers == gold_answers
        )

        expand_branch(
            branch=chosen_branch,
            visited=visited,
            node_depth=node_depth,
            expanded_branches=
                expanded_branches,
        )

        actions += 1

        retrieved_edges += (
            chosen_branch["target_count"]
        )

        history_relations.append(
            relation_name(
                chosen_branch["relation"]
            )
        )

        found_answers = (
            visited.intersection(
                gold_answers
            )
        )

        if was_complete_before_action:
            over_search_actions += 1

        if len(found_answers) > found_before:
            zero_gain_streak = 0
        else:
            zero_gain_streak += 1

        if found_answers == gold_answers:
            complete_seen = True

        if (
            policy == "patience1"
            and zero_gain_streak >= 1
        ):
            stop_reason = "patience_1"
            break

        if (
            policy == "patience2"
            and zero_gain_streak >= 2
        ):
            stop_reason = "patience_2"
            break

    if stop_reason is None:
        stop_reason = "action_budget"

    recall = (
        len(found_answers)
        / len(gold_answers)
    )

    precision = (
        1.0
        if found_answers
        else 0.0
    )

    if precision + recall == 0:
        answer_f1 = 0.0
    else:
        answer_f1 = (
            2.0
            * precision
            * recall
            / (precision + recall)
        )

    exact = int(
        found_answers == gold_answers
    )

    return {
        "qid": qid,
        "policy": policy,
        "gold_answer_count":
            len(gold_answers),
        "found_answer_count":
            len(found_answers),
        "answer_f1":
            float(answer_f1),
        "answer_recall":
            float(recall),
        "exact_match":
            exact,
        "actions":
            int(actions),
        "retrieved_edges":
            int(retrieved_edges),
        "premature":
            int(not exact),
        "over_search_actions":
            int(over_search_actions),
        "frontier_truncations":
            int(frontier_truncations),
        "total_available_frontiers":
            int(total_available_frontiers),
        "stop_reason": stop_reason,
        "last_maximum_positive_gain":
            maximum_stop_score,
    }


# ============================================================
# 11. RUN ONLINE TEST EVALUATION
# ============================================================

policies = [
    "fifo_fixed",
    "relation_fixed",
    "patience1",
    "patience2",
    "random_fixed",
    "model_fixed",
    "setstop",
    "oracle",
]

per_question_results = []

online_start = time.time()

for question_number, (
    qid,
    example
) in enumerate(
    test_examples,
    start=1
):
    for policy in policies:
        result = evaluate_search_policy(
            qid=qid,
            example=example,
            policy=policy,
        )

        per_question_results.append(
            result
        )

    if question_number % 25 == 0:
        elapsed_minutes = (
            time.time() - online_start
        ) / 60

        print(
            f"Online questions: "
            f"{question_number}/"
            f"{len(test_examples)} | "
            f"{elapsed_minutes:.1f} min"
        )


# ============================================================
# 12. BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================

def bootstrap_mean_ci(
    values,
    number_of_samples=2000,
    seed=42
):
    values = np.asarray(
        values,
        dtype=np.float64
    )

    rng = np.random.default_rng(seed)

    bootstrap_means = []

    for _ in range(number_of_samples):
        sample_indices = rng.integers(
            0,
            len(values),
            size=len(values),
        )

        bootstrap_means.append(
            np.mean(
                values[sample_indices]
            )
        )

    lower, upper = np.percentile(
        bootstrap_means,
        [2.5, 97.5]
    )

    return {
        "mean": float(
            np.mean(values)
        ),
        "ci95_lower": float(lower),
        "ci95_upper": float(upper),
    }


grouped_results = (
    collections.defaultdict(list)
)

for result in per_question_results:
    grouped_results[
        result["policy"]
    ].append(result)


online_summary = {}

metric_names = [
    "answer_f1",
    "answer_recall",
    "exact_match",
    "actions",
    "retrieved_edges",
    "premature",
    "over_search_actions",
    "frontier_truncations",
]

for policy, results in grouped_results.items():
    policy_summary = {
        "questions": len(results),
        "stop_reasons": dict(
            collections.Counter(
                result["stop_reason"]
                for result in results
            )
        ),
    }

    for metric_name in metric_names:
        values = [
            result[metric_name]
            for result in results
        ]

        policy_summary[metric_name] = (
            bootstrap_mean_ci(
                values=values,
                number_of_samples=2000,
                seed=stable_seed(
                    f"{policy}_{metric_name}"
                ),
            )
        )

    online_summary[policy] = (
        policy_summary
    )


# ============================================================
# 13. PAIRED SETSTOP COMPARISONS
# ============================================================

def paired_bootstrap_difference(
    first_values,
    second_values,
    number_of_samples=5000,
    seed=42
):
    first_values = np.asarray(
        first_values,
        dtype=np.float64
    )

    second_values = np.asarray(
        second_values,
        dtype=np.float64
    )

    differences = (
        first_values - second_values
    )

    rng = np.random.default_rng(seed)

    bootstrap_differences = []

    for _ in range(number_of_samples):
        indices = rng.integers(
            0,
            len(differences),
            size=len(differences),
        )

        bootstrap_differences.append(
            np.mean(
                differences[indices]
            )
        )

    lower, upper = np.percentile(
        bootstrap_differences,
        [2.5, 97.5]
    )

    return {
        "mean_difference": float(
            np.mean(differences)
        ),
        "ci95_lower": float(lower),
        "ci95_upper": float(upper),
        "significant_at_0_05": bool(
            lower > 0 or upper < 0
        ),
    }


results_by_policy_and_qid = {
    policy: {
        result["qid"]: result
        for result in results
    }
    for policy, results
    in grouped_results.items()
}

paired_comparisons = {}

for baseline_policy in [
    "fifo_fixed",
    "relation_fixed",
    "patience2",
    "random_fixed",
    "model_fixed",
]:
    paired_comparisons[
        baseline_policy
    ] = {}

    for metric_name in [
        "answer_f1",
        "answer_recall",
        "exact_match",
    ]:
        setstop_values = []
        baseline_values = []

        for qid, setstop_result in (
            results_by_policy_and_qid[
                "setstop"
            ].items()
        ):
            baseline_result = (
                results_by_policy_and_qid[
                    baseline_policy
                ][qid]
            )

            setstop_values.append(
                setstop_result[metric_name]
            )

            baseline_values.append(
                baseline_result[metric_name]
            )

        paired_comparisons[
            baseline_policy
        ][metric_name] = (
            paired_bootstrap_difference(
                first_values=
                    setstop_values,
                second_values=
                    baseline_values,
                seed=stable_seed(
                    f"{baseline_policy}_"
                    f"{metric_name}"
                ),
            )
        )


# ============================================================
# 14. SAVE RESULTS
# ============================================================

per_question_path = (
    OUTPUT_ROOT
    / "online_test_per_question_v1.jsonl"
)

with open(
    per_question_path,
    "w",
    encoding="utf-8"
) as file:
    for result in per_question_results:
        file.write(
            json.dumps(
                result,
                ensure_ascii=False
            )
            + "\n"
        )


online_summary_path = (
    OUTPUT_ROOT
    / "online_test_summary_v1.json"
)

online_summary_path.write_text(
    json.dumps(
        online_summary,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8",
)


paired_comparison_path = (
    OUTPUT_ROOT
    / "paired_bootstrap_comparisons_v1.json"
)

paired_comparison_path.write_text(
    json.dumps(
        paired_comparisons,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8",
)


complete_summary = {
    "evaluation_setting": (
        "oracle answer recognition; "
        "learned frontier ranking and stopping"
    ),
    "test_questions":
        len(test_examples),
    "frozen_stop_threshold":
        STOP_THRESHOLD,
    "maximum_actions":
        MAX_ACTIONS,
    "maximum_frontier_candidates":
        MAX_FRONTIER_CANDIDATES,
    "offline_test_metrics":
        offline_metrics,
    "online_test_summary":
        online_summary,
    "paired_comparisons":
        paired_comparisons,
    "runtime_minutes": float(
        (time.time() - online_start) / 60
    ),
}

complete_summary_path = (
    OUTPUT_ROOT
    / "complete_test_evaluation_v1.json"
)

complete_summary_path.write_text(
    json.dumps(
        complete_summary,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8",
)


# ============================================================
# 15. COMPACT RESULTS TABLE
# ============================================================

print("\n" + "=" * 110)
print("ONLINE UNTOUCHED-TEST RESULTS")
print("=" * 110)

header = (
    f"{'Policy':<18}"
    f"{'F1':>10}"
    f"{'Recall':>10}"
    f"{'Exact':>10}"
    f"{'Actions':>12}"
    f"{'Edges':>12}"
)

print(header)
print("-" * 110)

for policy in policies:
    result = online_summary[policy]

    print(
        f"{policy:<18}"
        f"{result['answer_f1']['mean']:>10.4f}"
        f"{result['answer_recall']['mean']:>10.4f}"
        f"{result['exact_match']['mean']:>10.4f}"
        f"{result['actions']['mean']:>12.2f}"
        f"{result['retrieved_edges']['mean']:>12.2f}"
    )

print("=" * 110)

print("\nOffline test candidate results:")
print(
    "Candidate accuracy:",
    round(
        offline_metrics[
            "candidate_accuracy"
        ],
        4
    )
)
print(
    "Candidate macro-F1:",
    round(
        offline_metrics[
            "candidate_macro_f1"
        ],
        4
    )
)
print(
    "Frozen-threshold STOP F1:",
    round(
        offline_metrics[
            "stop_f1"
        ],
        4
    )
)
print(
    "Premature-stop rate:",
    round(
        offline_metrics[
            "premature_stop_rate"
        ],
        4
    )
)

print("\nSaved files:")
print(offline_metrics_path)
print(per_question_path)
print(online_summary_path)
print(paired_comparison_path)
print(complete_summary_path)

print("\nIMPORTANT:")
print(
    "These results use oracle answer recognition. "
    "Gold answers identify whether a visited entity is "
    "an answer, but they are not supplied to the "
    "learned frontier scorer."
)

print("=" * 110)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 69.7 MB/s eta 0:00:00
Mounted at /content/drive
SETSTOP-KGQA UNTOUCHED TEST EVALUATION
Transformers: 5.17.0
PyTorch: 2.11.0+cu128
CUDA: True
Frozen threshold: 0.7100000000000001
Maximum actions: 12
Maximum frontier candidates: 16
GPU: Tesla T4


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Loaded trained controller:
/content/drive/MyDrive/SetStop_KGQA/models/setstop_qwen_gain_controller_v1

Running offline untouched-test candidate evaluation...


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Offline states: 500 | candidates: 7968
Offline states: 1000 | candidates: 15925
Offline states: 1500 | candidates: 23892
Offline states: 2000 | candidates: 31888
Offline states: 2500 | candidates: 39869
Offline states: 3000 | candidates: 47819
Offline states: 3500 | candidates: 55791
Offline states: 4000 | candidates: 63762
Offline states: 4500 | candidates: 71748
Offline states: 5000 | candidates: 79726
Offline states: 5500 | candidates: 87706
Offline states: 6000 | candidates: 95693

Offline untouched-test results
{
  "evaluation": "untouched_test_offline_states",
  "frozen_threshold": 0.7100000000000001,
  "number_of_candidates": 102401,
  "number_of_states": 6422,
  "candidate_accuracy": 0.7493481508969639,
  "candidate_balanced_accuracy": 0.7328774836067922,
  "candidate_macro_f1": 0.727139140949387,
  "candidate_weighted_f1": 0.7417418453289433,
  "classification_report": {
    "gain_0": {
      "precision": 0.7660201325804076,
      "recall": 0.877094343865962,
      "f1-score":